## Data Profiling and Catalog Audit
##### Objective: Perform an exploratory and data quality evaluation for a subset of products for Mexico to identify anomalies, missing values and inconsistencies that could represent an operative risk.

In [1]:
# load libraries and dataset
import pandas as pd
import numpy as np

df = pd.read_csv("../data/IKEA_product_catalog.csv")

df.head(3)

,unique_id,product_id,product_name,product_type,product_measurements,product_description,main_category,sub_category,product_rating,product_rating_count,badge,online_sellable,url,price,currency,discount,sale_tag,country
0,10018194-Australia,10018194,ORDNING,Dish drainer,50x27x36 cm,"ORDNING Dish drainer, stainless steel, 50x27x3...",kitchenware-tableware,dishwashing-accessories,3.3,140.0,none,True,https://www.ikea.com/au/en/p/ordning-dish-drai...,30.00,AUD,none,NONE,Australia
1,10018194-Austria,10018194,ORDNING,Dish drainer,50x27x36 cm,An ORDNING stainless-steel dish drainer with t...,kitchenware-tableware,dishwashing-accessories,3.3,6.0,none,True,https://www.ikea.com/at/en/p/ordning-dish-drai...,27.99,EUR,none,NONE,Austria
2,10018194-Bahrain,10018194,ORDNING,Dish drainer,50x27x36 cm,"ORDNING Dish drainer, stainless steel, 50x27x3...",kitchenware-tableware,dishwashing-accessories,1.0,1.0,none,True,https://www.ikea.com/bh/en/p/ordning-dish-drai...,9.50,BHD,none,NONE,Bahrain


In [2]:
# Create Mexico subset and save a base csv.
df_mex = df[df["country"] == "Mexico"].drop(columns=["country"]).reset_index(drop=True).copy()
df_mex.to_csv("../data/IKEA_product_catalog_MX.csv", index=False)

df_mex.head()

,unique_id,product_id,product_name,product_type,product_measurements,product_description,main_category,sub_category,product_rating,product_rating_count,badge,online_sellable,url,price,currency,discount,sale_tag
0,10018194-Mexico,10018194,ORDNING,Dish drainer,50x27x36 cm,"ORDNING Dish drainer, stainless steel, 50x27x3...",cookware-tableware,dishwashing-accessories,4.4,24.0,none,True,https://www.ikea.com/mx/en/p/ordning-dish-drai...,699.0,MXN,none,NONE
1,10056770-Mexico,10056770,DIMPA,Storage bag,65x22x65 cm,"DIMPA Storage bag, transparent, 65x22x65 cm",small-storage-organisers,clothes-organisers,4.8,75.0,none,True,https://www.ikea.com/mx/en/p/dimpa-storage-bag...,99.0,MXN,none,NONE
2,10069763-Mexico,10069763,OMAR,Shelving unit,92x36x94 cm,"OMAR Shelving unit, galvanised, 92x36x94 cm",storage-organisation,bookcases-shelving-units,4.8,71.0,none,True,https://www.ikea.com/mx/en/p/omar-shelving-uni...,1199.0,MXN,none,NONE
3,10091453-Mexico,10091453,TROFAST,Frame,99x44x94 cm,"TROFAST Frame, white, 99x44x94 cm",storage-organisation,storage-solution-systems,4.8,30.0,none,True,https://www.ikea.com/mx/en/p/trofast-frame-whi...,1699.0,MXN,none,NONE
4,10092216-Mexico,10092216,MJÖD,Beer tankard,60 cl,"MJÖD Beer tankard, clear glass, 60 cl",cookware-tableware,glassware-jugs,5.0,3.0,none,True,https://www.ikea.com/mx/en/p/mjoed-beer-tankar...,69.0,MXN,none,NONE


### Structural Diagnosys

In [3]:
df_mex.info()

<class 'pandas.DataFrame'>
RangeIndex: 7475 entries, 0 to 7474
Data columns (total 17 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   unique_id             7475 non-null   str    
 1   product_id            7475 non-null   int64  
 2   product_name          7475 non-null   str    
 3   product_type          7475 non-null   str    
 4   product_measurements  7475 non-null   str    
 5   product_description   7475 non-null   str    
 6   main_category         7475 non-null   str    
 7   sub_category          7475 non-null   str    
 8   product_rating        7475 non-null   str    
 9   product_rating_count  7475 non-null   str    
 10  badge                 7475 non-null   str    
 11  online_sellable       7475 non-null   bool   
 12  url                   7475 non-null   str    
 13  price                 7475 non-null   float64
 14  currency              7475 non-null   str    
 15  discount              7475 non-n

In [4]:
# explore columns for possible data types transformations.
for category in ["main_category", "sub_category", "badge", "online_sellable", "discount", "sale_tag"]:
        print(f'"{category}" columns contains {df_mex[category].nunique()} unique values.')

"main_category" columns contains 23 unique values.
"sub_category" columns contains 161 unique values.
"badge" columns contains 3 unique values.
"online_sellable" columns contains 2 unique values.
"discount" columns contains 1 unique values.
"sale_tag" columns contains 4 unique values.


In [5]:
# checking for unique ID's
df_mex["unique_id"].nunique()
df_mex["product_id"].nunique()
print(f"Total unique ID's: {df_mex["unique_id"].nunique()}\n"
      f"Total rows: {df_mex.shape[0]}")

Total unique ID's: 7475
Total rows: 7475


In [6]:
# Checking for anomalies in business logic

# Negative or equal to zero prices
if df_mex[df_mex["price"] <= 0.0].shape[0] == 0:
    print("No anomalies found in 'price' column.")

# check that all product measurements when included (value is not "none") are included in description
measure_in_description = df_mex.apply(lambda row: row["product_measurements"] in row["product_description"], axis=1)
none_description = df_mex[(~measure_in_description) & (df_mex["product_measurements"] != "none")]
if none_description.shape[0] == 0:
    print("No missing measurements.")

# product has a rating (less than or equal to 5) and has product ratings count > 0
# confirm that rating values are between 0 and 5
ratings = df_mex[df_mex["product_rating"] != "none"]["product_rating"].astype(float) # remove "none" values from subset to convert to float
if len(ratings[ratings.between(0, 5)]) == len(ratings):  # if both lens are equal it means all ratings are between 0 and 5
    print("All ratings between 0 and 5.")

# confirm that every row with a rating value has at least one review.
if df_mex[(df_mex["product_rating"] != "none") & (df_mex["product_rating_count"] == "none")].shape[0] == 0:
    print("All rating values have a rating count.")

No anomalies found in 'price' column.
No missing measurements.
All ratings between 0 and 5.
All rating values have a rating count.


## Data Profiling and Catalog Audit Analysis
From the **structural analysis** we can confirm that the dataset does not have missing values. Null values in critical columns like product_measurements or product_description can affect the customer experience or make it harder for the customers to find the right product for them.

The number of total unique IDs and product IDs match the total number of rows, which means there is a unique ID for every product and there are not repeated products.

For the **individual columns analysis** the following transformations might be considered:
- `"product_id"` data type "int64" into "str".
- `"main_category"` change data type "str" into "category". The columns consists of only 23 different values.
- `"badge"` change data type from "str" into "category".
- `"sale_tag"` change data type from "str" into "category".
- `"product_rating"` and `"product_rating_count"` change data types to float32. Transforming these columns into float values will enable calculations on such columns.
- Drop `"currency"` column. Working on a subset for a specific country makes a specific column for currency values unnecessary, dropping the column depends on the designated use of the dataset after the process (i.e. joining this subset to another country's dataset may require to stablish a currency value for each).
- Drop `"discount"` column since it contains no relevant values for this subset.

**Business logic dataset's analysis**
Dataset was checked for negative or equal to zero prices, missing or mismatched measurements and consistency between rating scores and number of ratings.
No inconsistencies were found. Having an inconsistency in any of the above mentioned might result in a negative experience for the customer and it also might cause problems for the customer to find the right products.